# PPO with a Recurrent Policy (GRU): Maze Navigation with Only Local Information

This notebook trains a PPO agent that can only see a small **local window** around itself in a maze — not its (row, col) position, not the full map. This turns the task into a **POMDP** (partially observable MDP): many interior corridor cells look identical from a purely local view, so a memoryless policy cannot tell them apart.

**What you will build, step by step:**
1. A maze environment that returns a local egocentric window instead of full state
2. Two policy architectures with the *same* PPO training loop: a **recurrent (GRU)** Actor-Critic that can integrate observations over time, and a **reactive (MLP)** Actor-Critic with no memory, for comparison
3. GAE, an episode-based rollout buffer (needed so the GRU's hidden state resets cleanly per episode), and the PPO clipped objective
4. A full training loop, run once per architecture
5. The head-to-head comparison: **does memory help when only local information is available?**
6. Evaluation: watch both trained agents solve the maze

This mirrors the `PPO/ppo_lunarlander.ipynb` notebook in structure (same PPO-clip / GAE / Actor-Critic machinery), with two changes: the network can be recurrent, and the buffer collects whole episodes instead of a fixed N-step window, since BPTT needs the sequence to stay in order.

## Cell 1 — Imports

In [ ]:
import random
import time
import warnings

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')

## Cell 2 — Partially Observable Maze Environment

Same grid encoding as `Qlearning/q_learning_maze.ipynb` (`0`=open, `1`=wall, `2`=goal), but the
agent no longer observes its `(row, col)`. Instead it observes a `(2R+1) x (2R+1)` **egocentric
window** centered on itself (cells outside the grid count as walls), plus a one-hot of its
last action. With `R=1` that's a 3x3 window flattened to 9 values, plus 4 for the last action
→ a 13-dimensional observation.

**Why this matters:** most open corridor cells in a 1-wide maze look *exactly* the same locally
(wall, wall, open ahead, open behind). A policy with no memory of where it has already been or
which way it came from has no way to break that symmetry other than trial and error — this is
the textbook motivation for a recurrent policy.

The maze below is a generated "perfect maze" (single solution path, no loops) with 1-wide
corridors, which maximizes exactly this kind of local aliasing.

**Reward structure (same as the Q-learning maze):** wall bump `-5`, valid step `-1`, goal `+100`.

In [ ]:
MAZE_GRID = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
    [1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 0, 0, 0, 0, 0, 2, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
]
START = (1, 1)   # shortest path to the goal is 20 steps


class PartialObsMazeEnv:
    """
    Grid maze identical to MazeEnv (Qlearning notebook), except the agent
    observes only a small local window around itself, not (row, col).

    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    """

    ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT = 0, 1, 2, 3
    ACTION_NAMES = ['Up', 'Down', 'Left', 'Right']
    ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    REWARD_GOAL = 100
    REWARD_STEP = -1
    REWARD_WALL = -5

    def __init__(self, grid, start, window_radius=1, max_steps=150):
        self.grid = np.array(grid)
        self.nrows, self.ncols = self.grid.shape
        self.start = start

        goal_positions = list(zip(*np.where(self.grid == 2)))
        assert len(goal_positions) == 1, 'Maze must have exactly one goal cell.'
        self.goal = tuple(int(x) for x in goal_positions[0])

        self.window_radius = window_radius
        self.max_steps = max_steps
        n_window_cells = (2 * window_radius + 1) ** 2
        self.obs_dim = n_window_cells + 4   # local window + one-hot(last action)
        self.n_actions = 4

    def _local_window(self, pos):
        """Egocentric (2R+1)x(2R+1) window; out-of-bounds cells count as walls."""
        r, c = pos
        R = self.window_radius
        window = np.ones((2 * R + 1, 2 * R + 1), dtype=np.float32)  # default = wall
        for dr in range(-R, R + 1):
            for dc in range(-R, R + 1):
                rr, cc = r + dr, c + dc
                if 0 <= rr < self.nrows and 0 <= cc < self.ncols:
                    window[dr + R, dc + R] = float(self.grid[rr, cc])
        return window.flatten() / 2.0   # scale {0,1,2} -> {0.0, 0.5, 1.0}

    def _make_obs(self):
        window = self._local_window(self.pos)
        last_action_onehot = np.zeros(4, dtype=np.float32)
        if self.last_action is not None:
            last_action_onehot[self.last_action] = 1.0
        return np.concatenate([window, last_action_onehot]).astype(np.float32)

    def reset(self):
        self.pos = self.start
        self.last_action = None
        self.steps = 0
        return self._make_obs()

    def step(self, action):
        dr, dc = self.ACTION_DELTAS[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc
        self.steps += 1

        if not (0 <= nr < self.nrows and 0 <= nc < self.ncols) or self.grid[nr, nc] == 1:
            reward, done = self.REWARD_WALL, False          # bumped a wall — stay in place
        elif (nr, nc) == self.goal:
            self.pos = (nr, nc)
            reward, done = self.REWARD_GOAL, True
        else:
            self.pos = (nr, nc)
            reward, done = self.REWARD_STEP, False

        if self.steps >= self.max_steps:
            done = True   # truncate a wandering episode

        self.last_action = action
        return self._make_obs(), reward, done


env = PartialObsMazeEnv(MAZE_GRID, START)
print(f'Maze size        : {env.nrows} x {env.ncols}')
print(f'Start / Goal     : {env.start} / {env.goal}')
print(f'Observation dim  : {env.obs_dim}  (3x3 local window + last-action one-hot)')
print(f'Action dim       : {env.n_actions}')

## Cell 3 — Visualize the Maze and What the Agent Actually Sees

In [ ]:
def plot_maze(env, title='Maze (full map — NOT what the agent sees)'):
    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = LinearSegmentedColormap.from_list('maze', ['white', 'black', 'limegreen'])
    ax.imshow(env.grid, cmap=cmap, vmin=0, vmax=2)
    for x in range(env.ncols + 1):
        ax.axvline(x - 0.5, color='gray', linewidth=0.8)
    for y in range(env.nrows + 1):
        ax.axhline(y - 0.5, color='gray', linewidth=0.8)
    for r in range(env.nrows):
        for c in range(env.ncols):
            if (r, c) == env.start:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=13, fontweight='bold', color='blue')
            elif (r, c) == env.goal:
                ax.text(c, r, 'G', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

plot_maze(env)

# Show what the agent's local observation looks like at a few points along the corridor
sample_positions = [(1, 1), (1, 5), (5, 1), (9, 5)]
fig, axes = plt.subplots(1, len(sample_positions), figsize=(3 * len(sample_positions), 3))
fig.suptitle('The agent only ever sees a 3x3 window like these — full position is never given', fontsize=11)
for ax, pos in zip(axes, sample_positions):
    env.pos = pos
    window = env._local_window(pos).reshape(3, 3)
    ax.imshow(window, cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(f'agent at {pos}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Cell 4 — Two Policy Architectures, One Interface

Both networks below expose the same three methods so the rest of the notebook (buffer, PPO
loss, training loop) doesn't need to know which one it's training:

- `init_hidden(batch_size)` — recurrent state to carry across a rollout (or `None`)
- `forward_step(obs, hidden)` — one environment step → `(logits, value, new_hidden)`
- `evaluate_sequence(obs_seq, actions_seq, hidden0)` — replays a whole stored episode for the
  PPO update, returning `(log_probs, values, entropy)` for every step

**`RecurrentActorCritic`** — a `GRUCell` carries a hidden state across the episode, so the
network can (in principle) count steps, remember which turns it already tried, and disambiguate
corridor cells that look locally identical.

**`ReactiveActorCritic`** — a plain 2-layer MLP baseline with no memory at all: same PPO
algorithm, same observation, only the architecture differs. Hidden size is chosen so both
networks have a comparable parameter count, so any difference in learning speed is about
memory, not raw capacity.

In [ ]:
class RecurrentActorCritic(nn.Module):
    """Shared GRUCell backbone with an actor head and a critic head."""

    def __init__(self, obs_dim, n_actions, hidden_size=64):
        super().__init__()
        self.hidden_size = hidden_size
        self.gru = nn.GRUCell(obs_dim, hidden_size)
        self.actor_head = nn.Linear(hidden_size, n_actions)
        self.critic_head = nn.Linear(hidden_size, 1)
        nn.init.orthogonal_(self.actor_head.weight, gain=0.01)   # near-uniform initial policy
        nn.init.orthogonal_(self.critic_head.weight, gain=1.0)

    def init_hidden(self, batch_size=1):
        return torch.zeros(batch_size, self.hidden_size, device=device)

    def forward_step(self, obs, hidden):
        hidden = self.gru(obs, hidden)
        logits = self.actor_head(hidden)
        value = self.critic_head(hidden).squeeze(-1)
        return logits, value, hidden

    def evaluate_sequence(self, obs_seq, actions_seq, hidden0):
        """Replay one full episode through the GRU to get gradients for PPO (BPTT)."""
        hidden = hidden0
        logits_list, values_list = [], []
        for t in range(obs_seq.shape[0]):
            logits, value, hidden = self.forward_step(obs_seq[t:t + 1], hidden)
            logits_list.append(logits)
            values_list.append(value)
        logits_all = torch.cat(logits_list, dim=0)
        values_all = torch.cat(values_list, dim=0)
        dist = Categorical(logits=logits_all)
        return dist.log_prob(actions_seq), values_all, dist.entropy()


class ReactiveActorCritic(nn.Module):
    """Memoryless MLP baseline — same interface, ignores/returns no hidden state."""

    def __init__(self, obs_dim, n_actions, hidden_size=110):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_size), nn.Tanh(),
            nn.Linear(hidden_size, hidden_size), nn.Tanh(),
        )
        self.actor_head = nn.Linear(hidden_size, n_actions)
        self.critic_head = nn.Linear(hidden_size, 1)
        nn.init.orthogonal_(self.actor_head.weight, gain=0.01)
        nn.init.orthogonal_(self.critic_head.weight, gain=1.0)

    def init_hidden(self, batch_size=1):
        return None

    def forward_step(self, obs, hidden):
        features = self.shared(obs)
        logits = self.actor_head(features)
        value = self.critic_head(features).squeeze(-1)
        return logits, value, hidden   # hidden passed through unchanged (None)

    def evaluate_sequence(self, obs_seq, actions_seq, hidden0):
        """No recurrence, so the whole episode can be evaluated in one vectorized pass."""
        features = self.shared(obs_seq)
        logits = self.actor_head(features)
        values = self.critic_head(features).squeeze(-1)
        dist = Categorical(logits=logits)
        return dist.log_prob(actions_seq), values, dist.entropy()


gru_net_check = RecurrentActorCritic(env.obs_dim, env.n_actions)
mlp_net_check = ReactiveActorCritic(env.obs_dim, env.n_actions)
print(f'RecurrentActorCritic (GRU) parameters : {sum(p.numel() for p in gru_net_check.parameters()):,}')
print(f'ReactiveActorCritic  (MLP) parameters : {sum(p.numel() for p in mlp_net_check.parameters()):,}')

## Cell 5 — Generalized Advantage Estimation (GAE)

Identical to `PPO/ppo_lunarlander.ipynb` — see that notebook for the derivation. Every maze
episode ends in a terminal state (goal reached or truncated at `max_steps`), so we always
bootstrap with `last_value=0`.

In [ ]:
def compute_gae(rewards, values, dones, last_value, gamma=0.99, gae_lambda=0.95):
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        next_value = last_value if t == T - 1 else values[t + 1]
        next_value = next_value * (1.0 - float(dones[t]))
        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * gae_lambda * (1.0 - float(dones[t])) * gae
        advantages[t] = gae
    returns = advantages + np.array(values, dtype=np.float32)
    return advantages, returns

## Cell 6 — Episode Buffer

`PPO/ppo_lunarlander.ipynb` uses a fixed `n_steps` rollout buffer that can span episode
boundaries — fine for a memoryless policy. A recurrent policy needs its hidden state reset at
the start of every episode and its steps replayed **in order** during the update (BPTT), so
here the buffer stores **whole episodes** instead, and the update loop iterates over complete
episodes rather than shuffled individual steps.

In [ ]:
class EpisodeBuffer:
    def __init__(self):
        self.reset()

    def reset(self):
        self.episodes = []

    def add_episode(self, obs, actions, log_probs, rewards, values, dones, gamma, gae_lambda):
        advantages, returns = compute_gae(rewards, values, dones, last_value=0.0,
                                           gamma=gamma, gae_lambda=gae_lambda)
        self.episodes.append(dict(obs=obs, actions=actions, log_probs=log_probs,
                                   advantages=advantages, returns=returns))

    def normalize_advantages(self):
        """Normalize using mean/std pooled across every episode in the buffer."""
        all_adv = np.concatenate([ep['advantages'] for ep in self.episodes])
        mean, std = all_adv.mean(), all_adv.std() + 1e-8
        for ep in self.episodes:
            ep['advantages'] = (ep['advantages'] - mean) / std

## Cell 7 — PPO Agent (works with either network)

Same PPO-clip loss as the LunarLander notebook, applied per-episode. `collect_rollout` runs
`n_episodes_per_iter` full episodes; `update` runs `k_epochs` passes over those episodes, each
time replaying the network through the whole stored sequence with `evaluate_sequence`.

In [ ]:
class PPOAgent:
    def __init__(self, network, n_episodes_per_iter=6, k_epochs=4, clip_epsilon=0.2,
                 gamma=0.99, gae_lambda=0.95, lr=3e-4, value_coef=0.5, entropy_coef=0.03,
                 max_grad_norm=0.5):
        self.net = network
        self.n_episodes_per_iter = n_episodes_per_iter
        self.k_epochs = k_epochs
        self.clip_epsilon = clip_epsilon
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        self.optimizer = optim.Adam(self.net.parameters(), lr=lr, eps=1e-5)
        self.buffer = EpisodeBuffer()
        self.policy_losses, self.value_losses, self.entropies = [], [], []

    def collect_rollout(self, env):
        """Phase 1: run n_episodes_per_iter full episodes with the current policy."""
        self.buffer.reset()
        ep_rewards, ep_lengths, ep_successes = [], [], []

        for _ in range(self.n_episodes_per_iter):
            obs = env.reset()
            hidden = self.net.init_hidden()
            obs_list, actions, log_probs, rewards, values, dones = [], [], [], [], [], []
            done = False
            while not done:
                obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    logits, value, hidden = self.net.forward_step(obs_t, hidden)
                    dist = Categorical(logits=logits)
                    action = dist.sample()
                    log_prob = dist.log_prob(action)
                next_obs, reward, done = env.step(action.item())

                obs_list.append(obs); actions.append(action.item())
                log_probs.append(log_prob.item()); rewards.append(reward)
                values.append(value.item()); dones.append(done)
                obs = next_obs

            ep_rewards.append(sum(rewards))
            ep_lengths.append(len(rewards))
            ep_successes.append(rewards[-1] == env.REWARD_GOAL)
            self.buffer.add_episode(obs_list, actions, log_probs, rewards, values, dones,
                                     self.gamma, self.gae_lambda)

        self.buffer.normalize_advantages()
        return ep_rewards, ep_lengths, ep_successes

    def update(self):
        """Phase 2: K epochs of PPO-clip updates, one full episode at a time."""
        episodes = list(self.buffer.episodes)
        for _ in range(self.k_epochs):
            random.shuffle(episodes)   # order across episodes; within-episode order is untouched
            for ep in episodes:
                hidden0 = self.net.init_hidden()
                obs_seq = torch.tensor(np.array(ep['obs']), dtype=torch.float32, device=device)
                actions_seq = torch.tensor(ep['actions'], dtype=torch.long, device=device)
                old_log_probs = torch.tensor(ep['log_probs'], dtype=torch.float32, device=device)
                advantages = torch.tensor(ep['advantages'], dtype=torch.float32, device=device)
                returns = torch.tensor(ep['returns'], dtype=torch.float32, device=device)

                new_log_probs, values, entropy = self.net.evaluate_sequence(obs_seq, actions_seq, hidden0)

                ratio = torch.exp(new_log_probs - old_log_probs)
                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                value_loss = F.mse_loss(values, returns)
                entropy_mean = entropy.mean()
                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy_mean

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), self.max_grad_norm)
                self.optimizer.step()

                self.policy_losses.append(policy_loss.item())
                self.value_losses.append(value_loss.item())
                self.entropies.append(entropy_mean.item())

## Cell 8 — Train: Recurrent (GRU) PPO vs Reactive (MLP) PPO

Same environment, same PPO hyperparameters, same random seed — the only thing that changes is
the network architecture. This is the actual experiment: **does giving the policy memory help
it solve a maze it can only see one 3x3 window of at a time?**

In [ ]:
HP = dict(
    n_episodes_per_iter = 6,
    k_epochs            = 4,
    clip_epsilon         = 0.2,
    gamma                 = 0.99,
    gae_lambda            = 0.95,
    lr                    = 3e-4,
    value_coef            = 0.5,
    entropy_coef          = 0.03,
    max_grad_norm         = 0.5,
)
N_ITERATIONS = 200


def train(network, n_iterations, label):
    agent = PPOAgent(network, **HP)
    history = {'success': [], 'reward': [], 'length': []}
    t_start = time.time()
    for it in range(1, n_iterations + 1):
        ep_r, ep_l, ep_s = agent.collect_rollout(env)
        agent.update()
        history['success'].extend(ep_s)
        history['reward'].extend(ep_r)
        history['length'].extend(ep_l)
        if it % 25 == 0:
            recent = slice(-HP['n_episodes_per_iter'] * 10, None)   # last ~10 iterations
            print(f'[{label}] iter {it:4d}/{n_iterations}  '
                  f'success_rate(last~60ep)={np.mean(history["success"][recent]):.2f}  '
                  f'mean_reward={np.mean(history["reward"][recent]):7.1f}  '
                  f'mean_len={np.mean(history["length"][recent]):6.1f}  '
                  f'({time.time() - t_start:.0f}s elapsed)')
    return agent, history


torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
env = PartialObsMazeEnv(MAZE_GRID, START)
gru_net = RecurrentActorCritic(env.obs_dim, env.n_actions)
gru_agent, gru_history = train(gru_net, N_ITERATIONS, 'GRU')

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
env = PartialObsMazeEnv(MAZE_GRID, START)
mlp_net = ReactiveActorCritic(env.obs_dim, env.n_actions)
mlp_agent, mlp_history = train(mlp_net, N_ITERATIONS, 'MLP')

## Cell 9 — Training Curves: Does Memory Help?

In [ ]:
def rolling(data, window=60):
    if len(data) < window:
        return np.array(data, dtype=float)
    return np.convolve(data, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('Recurrent (GRU) vs Reactive (MLP) PPO — Partially Observable Maze', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(rolling(gru_history['success']) * 100, color='steelblue', linewidth=2, label='GRU (recurrent)')
ax.plot(rolling(mlp_history['success']) * 100, color='coral', linewidth=2, label='MLP (reactive)')
ax.set_title('Success Rate (rolling 60-episode)'); ax.set_xlabel('Episode'); ax.set_ylabel('%')
ax.set_ylim(0, 105); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(rolling(gru_history['reward']), color='steelblue', linewidth=2, label='GRU (recurrent)')
ax.plot(rolling(mlp_history['reward']), color='coral', linewidth=2, label='MLP (reactive)')
ax.set_title('Episode Reward (rolling 60-episode)'); ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(rolling(gru_history['length']), color='steelblue', linewidth=2, label='GRU (recurrent)')
ax.plot(rolling(mlp_history['length']), color='coral', linewidth=2, label='MLP (reactive)')
ax.axhline(20, color='green', linestyle='--', alpha=0.6, label='Shortest path (20)')
ax.set_title('Episode Length (rolling 60-episode)'); ax.set_xlabel('Episode'); ax.set_ylabel('Steps')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Final success rate (last 60 episodes) — GRU: {np.mean(gru_history["success"][-60:]):.0%}  '
      f'|  MLP: {np.mean(mlp_history["success"][-60:]):.0%}')

## Cell 10 — Evaluate: Watch Both Trained Agents Solve the Maze

Greedy rollout (`argmax` over action logits, no sampling) for each trained network, tracing the
path taken from start to goal.

In [ ]:
def run_greedy(env, net, max_steps=150):
    obs = env.reset()
    hidden = net.init_hidden()
    path = [env.pos]
    total_reward = 0
    for _ in range(max_steps):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, _, hidden = net.forward_step(obs_t, hidden)
            action = logits.argmax(dim=-1).item()
        obs, reward, done = env.step(action)
        path.append(env.pos)
        total_reward += reward
        if done:
            break
    reached_goal = env.pos == env.goal
    return path, total_reward, reached_goal


def plot_path(env, path, title):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    display_grid = np.where(env.grid == 1, -1, 0).astype(float)
    display_grid[env.goal] = 1
    cmap = LinearSegmentedColormap.from_list('maze', ['black', 'lightyellow', 'limegreen'])
    ax.imshow(display_grid, cmap=cmap, vmin=-1, vmax=1)
    for i, (r, c) in enumerate(path):
        color = 'blue' if i == 0 else ('green' if i == len(path) - 1 else 'royalblue')
        ax.add_patch(plt.Circle((c, r), 0.22, color=color, alpha=0.8, zorder=4))
    for i in range(len(path) - 1):
        r1, c1 = path[i]; r2, c2 = path[i + 1]
        ax.plot([c1, c2], [r1, r2], 'b--', linewidth=1.2, alpha=0.5, zorder=3)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()


env_eval = PartialObsMazeEnv(MAZE_GRID, START)
gru_path, gru_reward, gru_success = run_greedy(env_eval, gru_net)
mlp_path, mlp_reward, mlp_success = run_greedy(env_eval, mlp_net)

print(f'GRU greedy rollout : {len(gru_path)-1} steps, reward={gru_reward}, reached goal={gru_success}')
print(f'MLP greedy rollout : {len(mlp_path)-1} steps, reward={mlp_reward}, reached goal={mlp_success}')

plot_path(env_eval, gru_path, f'GRU (recurrent) — {len(gru_path)-1} steps, success={gru_success}')
plot_path(env_eval, mlp_path, f'MLP (reactive) — {len(mlp_path)-1} steps, success={mlp_success}')

## Summary and Next Steps

| Component | Implementation |
|-----------|---------------|
| `PartialObsMazeEnv` | Grid maze returning a local egocentric window instead of full state |
| `RecurrentActorCritic` | `GRUCell`-based Actor-Critic — carries hidden state across a rollout |
| `ReactiveActorCritic` | Memoryless MLP Actor-Critic — same interface, no hidden state |
| `compute_gae()` | Identical GAE from the LunarLander PPO notebook |
| `EpisodeBuffer` | Stores whole episodes (not an arbitrary N-step window) so BPTT stays in order |
| `PPOAgent` | Same PPO-clip loss, generic over both network types |
| Training | Same seed, same hyperparameters, only the architecture differs |

**Takeaway:** under identical partial observations, the recurrent policy has a structural
capability the reactive one lacks — it can integrate a history of local views into something
like an implicit position estimate — and the training curves above show what that's worth.

**Where this connects to the connectome-constrained RL idea:** the fly's actual navigation
circuit (antennal lobe → central complex → motor output) is also recurrent — the ellipsoid
body's heading-integration mechanism is a ring attractor, not a feedforward pass. A natural
next step is to replace `RecurrentActorCritic`'s `GRUCell` with a policy whose connectivity
(including recurrence) is constrained to match real central-complex wiring pulled from the
FlyWire/hemibrain connectome, and compare it against both baselines here.

**Other next steps:**
- Try `window_radius=0` (agent only sees the current cell) to make the POMDP harder and widen the gap
- Track approximate KL / clip fraction per update, as in the LunarLander notebook, to sanity-check training health
- Try an LSTM in place of the GRU and compare